In [88]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display, HTML

In [89]:
df_raw=pd.read_csv(r"C:\Users\HomePC\Desktop\EDA\processed_data_phase1.csv")
df = df_raw.copy()


In [90]:

# Re-apply dtypes lost in the CSV round-trip from Phase 1
df["date_recorded"] = pd.to_datetime(df["date_recorded"], errors="coerce")

for col in ["id", "region_code", "district_code"]:
    df[col] = df[col].astype("string")

categorical_cols = [
    "funder", "installer", "wpt_name", "basin", "subvillage", "region",
    "lga", "ward", "recorded_by", "scheme_management",
    "extraction_type", "extraction_type_group", "extraction_type_class",
    "management", "management_group", "payment", "payment_type",
    "water_quality", "quality_group", "quantity", "quantity_group",
    "source", "source_type", "source_class",
    "waterpoint_type", "waterpoint_type_group",
    "public_meeting", "permit"  # now 3-level string categories, not boolean
]
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype("category")

print(df.dtypes)

id                                       string
amount_tsh                              float64
date_recorded                    datetime64[us]
funder                                 category
gps_height                                int64
installer                              category
longitude                               float64
latitude                                float64
wpt_name                               category
num_private                               int64
basin                                  category
subvillage                             category
region                                 category
region_code                              string
district_code                            string
lga                                    category
ward                                   category
population                                int64
public_meeting                         category
recorded_by                            category
scheme_management                      c



WE GO FIX STUCTURAL ERRORS AND STANDARDIZE CATEGORIES

Here is where structural oddities get caught 

In [91]:
## Fix Structural Errors and Standardize Categories

# ---  Generic whitespace trim across text-like columns (safety net) ---
text_cols = df.select_dtypes(include="object").columns
text_cols

Index([], dtype='str')

In [92]:
for col in text_cols:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace("nan", pd.NA)

In [93]:
## This cleans the installer and funder columns so names 
##differing only by capitalization or extra spaces become one category##
before_installer = df["installer"].nunique()
before_funder = df["funder"].nunique()

for col in ["installer", "funder"]:
    df[col] = df[col].astype(str).str.strip().str.title()
    df[col] = df[col].replace("Nan", pd.NA)
    df[col] = df[col].astype("category")

print(f"installer: {before_installer} -> {df['installer'].nunique()} unique")
print(f"funder: {before_funder} -> {df['funder'].nunique()} unique")

installer: 2140 -> 1932 unique
funder: 1894 -> 1894 unique


In [94]:
# --- 10.4 Row/column count integrity check ---
assert len(df) == len(df_raw), "Row count changed during structural cleanup!"
print(f"Shape unchanged: {df.shape}")

Shape unchanged: (59400, 44)


In [95]:
# This checks for impossible dates: a water point should not be recorded before it was built.
future_construction = (
    (df["construction_year"]) != 0) & (df["construction_year"] > (df["date_recorded"].dt.year)
                                     )
print("construction_year after date_recorded:", future_construction.sum())    

construction_year after date_recorded: 9


In [96]:
# This checks whether any water points claim to have been built in a future year
import datetime
current_year = datetime.date.today().year
future_vs_today = (df["construction_year"] != 0) & (df["construction_year"] > current_year)
print("construction_year in the future (vs today):", future_vs_today.sum())

construction_year in the future (vs today): 0


In [97]:
# This checks whether each water point’s coordinates fall inside Tanzania’s approximate geographic boundaries
# (not just the generic -90/90, -180/180 global range)
lat_min, lat_max = -11.75, -0.95
lon_min, lon_max = 29.3, 40.5

bad_lat = ~df["latitude"].between(lat_min, lat_max)
bad_lon = ~df["longitude"].between(lon_min, lon_max)
bad_geo = bad_lat | bad_lon

print("Out-of-bounds latitude:", bad_lat.sum())
print("Out-of-bounds longitude:", bad_lon.sum())
print("Rows with any bad coordinate:", bad_geo.sum())

Out-of-bounds latitude: 1812
Out-of-bounds longitude: 1812
Rows with any bad coordinate: 1812


In [112]:
display(HTML("""
<h1 style="text-align:center; color:red; font-size:50 px;">
     Issue yangu iko hapa madam hasa After handling missing values, wakati wa kucheki logical consistency am finding some values that are still invalid kwa mfano hzo hapo juu  values that do not make logical sense.

In such cases, is it okay to replace those values with "unknown" missing values, or should I handle them differently depending on the column?
</h1>
"""))

In [99]:
#  gps_height plausibility check ---
# Tanzania's elevation ranges from below sea level near the Rift Valley
# to Kilimanjaro (~5,895m) — flag anything outside a generous plausible range
implausible_height = ~df["gps_height"].between(-200, 6000)
print("Implausible gps_height:", implausible_height.sum())
print("Actual min/max:", df["gps_height"].min(), "/", df["gps_height"].max())

Implausible gps_height: 0
Actual min/max: -90 / 2770


In [100]:
#  Non-negativity checks on fields that should never be negative ---
for col in ["amount_tsh", "population", "num_private"]:
    n_negative = (df[col] < 0).sum()
    print(f"{col}: {n_negative} negative values")

amount_tsh: 0 negative values
population: 0 negative values
num_private: 0 negative values


In [101]:
# Dependent-field agreement: region_code should map to exactly one region ---
region_code_map = df.groupby("region_code")["region"].nunique()
inconsistent_codes = region_code_map[region_code_map > 1]
print(f"region_code values mapping to multiple region names: {len(inconsistent_codes)}")
if len(inconsistent_codes) > 0:
    for code in inconsistent_codes.index:
        print(f"  region_code {code}: {df.loc[df['region_code']==code, 'region'].unique()}")

region_code values mapping to multiple region names: 5
  region_code 11: ['Iringa', 'Shinyanga']
Categories (21, str): ['Arusha', 'Dar es Salaam', 'Dodoma', 'Iringa', ..., 'Shinyanga', 'Singida', 'Tabora', 'Tanga']
  region_code 14: ['Tabora', 'Shinyanga']
Categories (21, str): ['Arusha', 'Dar es Salaam', 'Dodoma', 'Iringa', ..., 'Shinyanga', 'Singida', 'Tabora', 'Tanga']
  region_code 17: ['Shinyanga', 'Mwanza']
Categories (21, str): ['Arusha', 'Dar es Salaam', 'Dodoma', 'Iringa', ..., 'Shinyanga', 'Singida', 'Tabora', 'Tanga']
  region_code 18: ['Kagera', 'Lindi']
Categories (21, str): ['Arusha', 'Dar es Salaam', 'Dodoma', 'Iringa', ..., 'Shinyanga', 'Singida', 'Tabora', 'Tanga']
  region_code 5: ['Morogoro', 'Tanga']
Categories (21, str): ['Arusha', 'Dar es Salaam', 'Dodoma', 'Iringa', ..., 'Shinyanga', 'Singida', 'Tabora', 'Tanga']
